# YOLO Object Detection — IRRG Tile Preprocessing

Prepares the IRRG (Infrared-Red-Green) detection dataset for YOLOv11 training.

**Pipeline**
1. Load raw RGB and CIR GeoTIFF tiles and tree-crown ground-truth polygons.
2. Build an IRRG composite (IR, R, G bands) in memory for each tile.
3. Slide a 512 × 512 window across each composite with split-specific overlap.
4. For each patch: clip crown annotations, convert to YOLO bbox format, save files.
5. Background patches sub-sampled to 10 % for training; val/test positive only.
6. Quercus capped at 3 000 training annotations.


**Import libraries**

In [ ]:
import math
import os
import random
import shutil
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, List

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.windows import Window
from shapely.geometry import box
from shapely.validation import make_valid

# Locate project root (works from any sub-directory inside the repo)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR


In [ ]:
random.seed(42)

**Define paths**

In [ ]:
# Ground-truth crowns for the detection task (one row per tree crown,
# with a 'species_na' column used to derive class IDs)
YOLO_DETECTION_GROUND_TRUTH = (
    INTERIM_DIR
    / "yolo_detection_gt_prep"
    / "enschede_yolo_detection_ground_truth_v01.shp"
)

# Directory that contains the raw RGB GeoTIFF tiles (RGB_34FN2.tif, etc.)
RGB_TILES_DIR = RAW_DIR / "rgb_tiles"

# Directory that contains the raw CIR GeoTIFF tiles (CIR_34FN2.tif, etc.)
CIR_TILES_DIR = RAW_DIR / "cir_tiles"

# Output root: train/, val/, and test/ sub-directories will be created here
IRRG_SPLITS_DIR = PROCESSED_DIR / "splits_irrg"


**Define constants**

In [ ]:
# Raster tile IDs — one per geographic area, mapped to a train/val/test split
RASTER_IDS = ["34FN2", "34FZ2", "35AN1"]

# Square patch size in pixels
TILE_SIZE = 512

# Fractional overlap between adjacent patches, per split.
# Training uses 50 % overlap to generate more samples from limited data;
# validation and test use 0 % to avoid evaluating the same pixels twice.
OVERLAP_BY_SPLIT = {
    "train": 0.5,
    "val":   0.0,
    "test":  0.0,
}

# Which geographic tile belongs to which split
SPLIT_MAP = {
    "34FN2": "train",
    "34FZ2": "val",
    "35AN1": "test",
}

# Fraction of background (no-crown) patches kept for training.
# Keeping all of them would flood the dataset with negatives; 10 % is a
# practical balance between class diversity and training efficiency.
TRAIN_BACKGROUND_RATIO = 0.10

# Cap on Quercus annotations in the training split.
# Quercus is by far the most common species; without a cap the model
# could over-fit to it and under-perform on rarer species.
MAX_QUERCUS = 3000
species_counter = {"Quercus": 0}


**Helper: convert band to uint8**

In [ ]:
def convert_to_uint8(band: np.ndarray) -> np.ndarray:
    """
    Normalise a single raster band to the 0-255 uint8 range.

    The band is first linearly rescaled so that its minimum maps to 0 and its
    maximum maps to 1, then multiplied by 255 and cast to uint8.  This makes
    the dynamic range of different tiles comparable regardless of their
    original radiometric values.

    Args:
        band: 2-D NumPy array of any numeric dtype.

    Returns:
        2-D NumPy array of dtype uint8.
    """
    band = (band - band.min()) / (band.max() - band.min())
    return (band * 255).astype("uint8")


**Build IRRG composite in memory**

In [ ]:
def stack_irrg(rgb_path: str, cir_path: str) -> tuple:
    """
    Open RGB and CIR GeoTIFFs, extract the R, G, and Near-Infrared (NIR) bands,
    convert each to uint8, and return the stacked array with rasterio metadata.

    The IRRG (Infrared-Red-Green) composite highlights vegetation because
    chlorophyll reflects strongly in the NIR range, making tree crowns easier
    to separate from urban surfaces.

    Args:
        rgb_path: File path to a 3-band RGB GeoTIFF (band 1 = R, band 2 = G).
        cir_path: File path to a 3-band CIR GeoTIFF (band 1 = NIR).

    Returns:
        stacked: NumPy array of shape (3, H, W) and dtype uint8 (IR, R, G).
        meta:    Rasterio metadata dict (from the RGB file) used when writing
                 patch GeoTIFFs.
        rgb_obj: The open RGB rasterio DatasetReader.
        cir_obj: The open CIR rasterio DatasetReader.
    """
    rgb_obj = rasterio.open(rgb_path)
    cir_obj = rasterio.open(cir_path)
    R   = convert_to_uint8(rgb_obj.read(1))
    G   = convert_to_uint8(rgb_obj.read(2))
    IR  = convert_to_uint8(cir_obj.read(1))
    stacked = np.stack([IR, R, G], axis=0)
    meta = rgb_obj.meta.copy()
    meta.update({"driver": "GTiff", "count": 3, "dtype": "uint8"})
    return stacked, meta, rgb_obj, cir_obj


**Generate YOLO bounding-box label for one crown**

In [ ]:
def create_yolo_label(
    crown_row,
    src_transform,
    x_off: int,
    y_off: int,
    w: int,
    h: int,
    class_id: int,
) -> str | None:
    """
    Convert a single tree-crown polygon into a YOLO bounding-box annotation.

    The annotation format is:
        <class_id> <x_center> <y_center> <box_width> <box_height>
    where all four coordinates are normalised to [0, 1] relative to the patch
    dimensions.

    A crown is rejected (returns None) if its bounding box touches or crosses
    the patch edge — partial crowns at boundaries are excluded to avoid
    misleading the detector with truncated objects.

    Args:
        crown_row:     Row from the tree-crown GeoDataFrame.
        src_transform: Affine transform of the source raster.
        x_off, y_off:  Top-left pixel offset of the patch within the raster.
        w, h:          Width and height of the patch in pixels.
        class_id:      Integer class ID assigned to this species.

    Returns:
        Formatted YOLO label string, or None if the box should be discarded.
    """
    geom = crown_row.geometry
    minx, miny, maxx, maxy = geom.bounds

    # Convert map coordinates → floating-point pixel coordinates
    px_min, py_min = ~src_transform * (minx, miny)
    px_max, py_max = ~src_transform * (maxx, maxy)

    # Express in local patch pixel coordinates
    xmin = px_min - x_off
    xmax = px_max - x_off
    ymin = py_max - y_off  # raster y-axis is inverted relative to map y
    ymax = py_min - y_off

    # Discard crowns whose bounding box touches or exceeds the patch boundary
    if xmin <= 0 or ymin <= 0 or xmax >= w or ymax >= h:
        return None

    # Clamp to patch bounds (safety net after the reject test above)
    xmin = max(0, min(xmin, w))
    xmax = max(0, min(xmax, w))
    ymin = max(0, min(ymin, h))
    ymax = max(0, min(ymax, h))

    x_center = (xmin + xmax) / (2.0 * w)
    y_center = (ymin + ymax) / (2.0 * h)
    box_w    = (xmax - xmin) / w
    box_h    = (ymax - ymin) / h

    if box_w <= 0 or box_h <= 0:
        return None

    return f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}"


**Save background patch (if allowed by split rules)**

In [ ]:
def save_background_if_allowed(
    src,
    window: Window,
    tile_id: str,
    images_dir: str,
    split_name: str,
) -> int:
    """
    Optionally save a background (no-crown) patch to disk.

    Background patches are never saved for val/test splits because those splits
    are used exclusively for evaluation on positive examples.  For the training
    split a random sub-sample controlled by TRAIN_BACKGROUND_RATIO is kept; this
    prevents the negative class from dominating the training data.

    Args:
        src:        Open rasterio DatasetReader for the stacked raster.
        window:     Rasterio Window describing the patch extent.
        tile_id:    Filename stem (no extension) for the output patch.
        images_dir: Directory where the patch GeoTIFF will be written.
        split_name: One of 'train', 'val', or 'test'.

    Returns:
        0 (always, so callers can accumulate the return value without special
        handling for background patches).
    """
    if split_name in ("val", "test"):
        return 0
    if split_name == "train" and random.random() > TRAIN_BACKGROUND_RATIO:
        return 0

    tile_data       = src.read(window=window)
    tile_transform  = rasterio.windows.transform(window, src.transform)
    patch_path      = os.path.join(images_dir, f"{tile_id}.tif")

    with rasterio.open(
        patch_path, "w",
        driver="GTiff",
        height=window.height,
        width=window.width,
        count=src.count,
        dtype="uint8",
        crs=src.crs,
        transform=tile_transform,
    ) as dst:
        dst.write(tile_data)

    return 0


**Process one patch and write image + label files**

In [ ]:
def process_patch_and_annotate(
    src,
    crowns_vector,
    crowns_sindex,
    class_id_map: Dict[str, int],
    x_off: int,
    y_off: int,
    w: int,
    h: int,
    split_name: str,
    tile_id: str,
    images_dir: str,
    labels_dir: str,
) -> int:
    """
    Extract one image patch from the raster, generate YOLO bounding-box labels
    for all fully-contained tree crowns, and write the image and label files.

    Patches that extend beyond the raster boundary (edge padding) are silently
    discarded.  Patches where crowns intersect but none is fully inside are also
    discarded to avoid partially-labelled training examples.

    Args:
        src:            Open rasterio DatasetReader for the stacked raster.
        crowns_vector:  GeoDataFrame of crown polygons reprojected to the raster CRS.
        crowns_sindex:  Spatial index over crowns_vector for fast candidate lookup.
        class_id_map:   Mapping from species name string to integer class ID.
        x_off, y_off:   Top-left pixel offset of the patch within the raster.
        w, h:           Patch width and height in pixels.
        split_name:     One of 'train', 'val', or 'test'.
        tile_id:        Filename stem for the output patch (image + label).
        images_dir:     Directory where the patch GeoTIFF is written.
        labels_dir:     Directory where the YOLO .txt label is written.

    Returns:
        Number of bounding-box annotations written for this patch.
    """
    global species_counter, MAX_QUERCUS

    window = Window(x_off, y_off, w, h)

    # Skip patches that fall partially outside the raster
    if x_off + w > src.width or y_off + h > src.height:
        return 0

    tile_bounds = rasterio.windows.bounds(window, src.transform)
    tile_geom   = box(*tile_bounds)

    # Candidate crowns whose bounding boxes overlap the patch (fast pre-filter)
    possible_ids    = list(crowns_sindex.intersection(tile_geom.bounds))
    if not possible_ids:
        return save_background_if_allowed(src, window, tile_id, images_dir, split_name)

    possible_crowns  = crowns_vector.iloc[possible_ids]
    crowns_intersect = possible_crowns[possible_crowns.geometry.intersects(tile_geom)]
    crowns_full      = crowns_intersect[crowns_intersect.geometry.within(tile_geom)]

    has_any_crown  = len(crowns_intersect) > 0
    has_full_crowns = len(crowns_full) > 0

    # Crowns exist but all are clipped at the patch edge → discard
    if has_any_crown and not has_full_crowns:
        return 0

    if not has_any_crown:
        return save_background_if_allowed(src, window, tile_id, images_dir, split_name)

    # Check Quercus cap before writing anything to disk
    species_in_patch = list(crowns_full["species_na"])
    only_quercus     = all(s == "Quercus" for s in species_in_patch)
    if split_name == "train" and only_quercus:
        if species_counter["Quercus"] >= MAX_QUERCUS:
            return 0

    tile_data       = src.read(window=window)
    patch_path      = os.path.join(images_dir, f"{tile_id}.tif")
    label_path      = os.path.join(labels_dir, f"{tile_id}.txt")
    tile_transform  = rasterio.windows.transform(window, src.transform)

    with rasterio.open(
        patch_path, "w",
        driver="GTiff",
        height=h,
        width=w,
        count=src.count,
        dtype="uint8",
        crs=src.crs,
        transform=tile_transform,
    ) as dst:
        dst.write(tile_data)

    labels = []
    for _, row in crowns_full.iterrows():
        species_name = row["species_na"].strip()

        if split_name == "train" and species_name == "Quercus":
            if species_counter["Quercus"] >= MAX_QUERCUS:
                continue
            species_counter["Quercus"] += 1

        class_id = class_id_map[species_name]
        ann = create_yolo_label(row, src.transform, x_off, y_off, w, h, class_id)
        if ann:
            labels.append(ann)

    if labels:
        with open(label_path, "w") as f:
            f.write("\n".join(labels))
        return len(labels)
    else:
        os.remove(patch_path)
        return 0


**Main loop: generate all patches and labels**

In [ ]:
def generate_patch_labels(
    raster_ids: List[str],
    crowns_gdf,
    split_mapping: Dict[str, str],
    splits_dir: str,
    tile_size: int,
) -> int:
    """
    Iterate over all raster tiles, build the IRRG composite entirely in memory
    using rasterio.MemoryFile, slide a window across it, and call
    process_patch_and_annotate() for every window position.

    rasterio.MemoryFile acts like an in-memory file so no intermediate GeoTIFF
    is ever written to disk — the stacked array goes straight from numpy into a
    MemoryFile and is read back from there by the patch loop.

    Args:
        raster_ids:    List of tile IDs (e.g. ['34FN2', '34FZ2', '35AN1']).
        crowns_gdf:    GeoDataFrame of tree crown polygons with 'species_na' column.
        split_mapping: Maps each tile ID to 'train', 'val', or 'test'.
        splits_dir:    Root output directory; images/ and labels/ are created inside.
        tile_size:     Side length of square patches in pixels.

    Returns:
        Total number of bounding-box annotations produced across all patches.
    """
    total_boxes  = 0
    species_list = sorted(crowns_gdf["species_na"].unique())
    class_id_map = {name: idx for idx, name in enumerate(species_list)}

    for rid in raster_ids:
        rgb_path   = os.path.join(RGB_TILES_DIR, f"RGB_{rid}.tif")
        cir_path   = os.path.join(CIR_TILES_DIR, f"CIR_{rid}.tif")
        split_name = split_mapping[rid]

        overlap_percent = OVERLAP_BY_SPLIT[split_name]
        step_size       = int(tile_size * (1 - overlap_percent))

        images_dir = os.path.join(splits_dir, split_name, "images")
        labels_dir = os.path.join(splits_dir, split_name, "labels")
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)

        # Build the stacked composite entirely in memory
        stacked, meta, rgb_obj, cir_obj = stack_irrg(rgb_path, cir_path)
        rgb_obj.close()
        cir_obj.close()

        # Write into a rasterio MemoryFile — no disk I/O
        with rasterio.MemoryFile() as mem:
            with mem.open(**meta) as dataset:
                dataset.write(stacked)

            with mem.open() as src:
                width, height = src.width, src.height
                crowns_vector = crowns_gdf.to_crs(src.crs)
                crowns_sindex = crowns_vector.sindex

                n_cols = math.ceil((width  - tile_size) / step_size) + 1
                n_rows = math.ceil((height - tile_size) / step_size) + 1

                for i in range(n_rows):
                    for j in range(n_cols):
                        x_off   = j * step_size
                        y_off   = i * step_size
                        tile_id = f"IR-RG_{rid}_patch_{i}_{j}"

                        total_boxes += process_patch_and_annotate(
                            src, crowns_vector, crowns_sindex, class_id_map,
                            x_off, y_off, tile_size, tile_size,
                            split_name, tile_id, images_dir, labels_dir,
                        )

    return total_boxes


**Run**

In [ ]:
crowns_gdf = gpd.read_file(YOLO_DETECTION_GROUND_TRUTH)

total_boxes = generate_patch_labels(
    raster_ids    = RASTER_IDS,
    crowns_gdf    = crowns_gdf,
    split_mapping = SPLIT_MAP,
    splits_dir    = str(IRRG_SPLITS_DIR),
    tile_size     = TILE_SIZE,
)
print(f"Total bounding-box annotations generated: {total_boxes}")
